# Analisis Imagenes ACV

**Dataset:** ATLAS R2.0 (Anatomical Tracings of Lesions After Stroke, preprocesado · NITRC · ~9.7 GB)  
**Objetivo del notebook:** cumplir el mínimo de la sesión del 6 de mayo según la rúbrica:

1. Dataset accesible y cargado.
2. Auditoría del dataset: *shape*, tipos, faltantes, duplicados.
3. EDA **univariado** de la(s) variable(s) objetivo candidata(s) y de los predictores clave.

**Predictores clave (acordados en grupo):** demográficos (edad, sexo) y características de lesión (volumen, lateralidad).

**Variable objetivo:** se decide al final del notebook entre tres candidatas: volumen de lesión (regresión), máscara voxel-wise (segmentación), lateralidad (clasificación).

> Cada gráfico de este notebook responde una pregunta concreta enunciada en el encabezado. No hay gráficos decorativos.

## 1. Setup y configuración

**Único parámetro a editar antes de correr el notebook:** `DATA_ROOT` — la ruta absoluta de la carpeta donde descomprimiste `ATLAS_2.zip` en tu máquina.

In [ ]:
from pathlib import Path
import os, sys, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# EDITAR: ruta absoluta a la carpeta que contiene 'Training/', 'Testing/' y el CSV de metadata.
# Ejemplos:
#   Windows: r'C:\Users\DELL\Datasets\ATLAS_2'
#   Mac/Linux: '/home/usuario/datasets/ATLAS_2'
DATA_ROOT = Path(r'C:\Users\DELL\Datasets\ATLAS_2')
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

OUTPUT_DIR = Path('outputs')
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(parents=True, exist_ok=True)

# Tamaño de muestra para cómputo de features de lesión desde NIfTI (rápido en Día 1).
# El cómputo full sobre los ~955 sujetos se hace en el notebook 02.
SAMPLE_N = 60
RANDOM_SEED = 42

print('DATA_ROOT:', DATA_ROOT)
print('Existe:', DATA_ROOT.exists())

In [ ]:
# Sanity check: estructura esperada del dataset.
assert DATA_ROOT.exists(), f'No existe {DATA_ROOT}. Edita DATA_ROOT en la celda anterior.'

subdirs = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()]
csvs = list(DATA_ROOT.glob('*.csv'))
print('Subdirectorios:', subdirs)
print('CSV(s) en raíz:', [c.name for c in csvs])

## 2. Carga de metadata

El archivo de metadata oficial es `20220425_ATLAS_2.0_MetaData.csv` (o similar `*ATLAS*MetaData*.csv`). Lo localizamos automáticamente para que el notebook sea robusto a renombres.

In [ ]:
candidates = list(DATA_ROOT.glob('*MetaData*.csv')) + list(DATA_ROOT.glob('*metadata*.csv')) + list(DATA_ROOT.glob('*.csv'))
candidates = list({c.resolve() for c in candidates})
if not candidates:
    raise FileNotFoundError('No se encontró CSV de metadata en DATA_ROOT.')
META_CSV = candidates[0]
print('Metadata file:', META_CSV.name)

df = pd.read_csv(META_CSV)
df.head()

In [ ]:
# Normalizamos nombres de columnas para hacer el resto del notebook robusto.
df.columns = [c.strip() for c in df.columns]
print('Columnas detectadas:')
for c in df.columns:
    print(f'  - {c!r:<40} dtype={df[c].dtype}')

In [ ]:
# Mapeo flexible: detectamos los nombres reales de columnas clave (varían entre releases).
def _find(cols, *patterns):
    for p in patterns:
        for c in cols:
            if re.search(p, c, flags=re.I):
                return c
    return None

COL_SUBJECT = _find(df.columns, r'^subject', r'sub.?id', r'^id$')
COL_AGE     = _find(df.columns, r'^age')
COL_SEX     = _find(df.columns, r'^sex', r'gender')
COL_COHORT  = _find(df.columns, r'cohort', r'site')
COL_LESION_VOL = _find(df.columns, r'lesion.?vol', r'volume')

detected = {
    'subject': COL_SUBJECT,
    'age':     COL_AGE,
    'sex':     COL_SEX,
    'cohort':  COL_COHORT,
    'lesion_volume': COL_LESION_VOL,
}
print('Columnas mapeadas a roles del proyecto:')
for k, v in detected.items():
    print(f'  {k:<16}: {v}')

## 3. Auditoría del dataset

Pregunta del análisis: *¿con qué dataset estamos realmente trabajando — qué tan completo, qué tan duplicado, qué tan ruidoso?*

In [ ]:
print('Shape:', df.shape)
print(f'Sujetos únicos por {COL_SUBJECT!r}:', df[COL_SUBJECT].nunique() if COL_SUBJECT else 'N/A')
df.dtypes.to_frame('dtype')

In [ ]:
# Faltantes por columna (absoluto y %).
miss = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(2)
}).sort_values('n_missing', ascending=False)
miss.to_csv(OUTPUT_DIR / 'tables' / 'missingness.csv')
miss

In [ ]:
# Duplicados: por fila completa y por subject ID.
dup_full = df.duplicated().sum()
dup_subj = df.duplicated(subset=[COL_SUBJECT]).sum() if COL_SUBJECT else None
print(f'Filas duplicadas (todas las columnas): {dup_full}')
print(f'Subject IDs duplicados: {dup_subj}')

## 4. EDA univariado · Predictores demográficos

### 4.1 Edad

**Pregunta:** ¿cuál es la distribución etaria del cohorte y representa a la población diana (pacientes con ACV en Guatemala)?  
*Nota:* en ATLAS la edad está binada en intervalos de 5 años (deidentificación).

In [ ]:
if COL_AGE is None:
    print('No se detectó columna de edad.')
else:
    age = df[COL_AGE]
    print(age.describe(include='all'))
    fig, ax = plt.subplots(figsize=(8, 4))
    if pd.api.types.is_numeric_dtype(age):
        age.dropna().plot(kind='hist', bins=20, ax=ax, edgecolor='white')
        ax.set_xlabel('Edad (años o bin medio)')
    else:
        order = sorted(age.dropna().unique(), key=lambda x: str(x))
        age.value_counts().reindex(order).plot(kind='bar', ax=ax)
        ax.set_xlabel('Bin de edad')
        plt.xticks(rotation=45, ha='right')
    ax.set_ylabel('N sujetos')
    ax.set_title('Distribución de edad — cohorte ATLAS R2.0')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / '4_1_edad.png', dpi=120)
    plt.show()

### 4.2 Sexo

**Pregunta:** ¿hay desbalance por sexo que pueda introducir sesgo en un modelo posterior?

In [ ]:
if COL_SEX is None:
    print('No se detectó columna de sexo.')
else:
    counts = df[COL_SEX].value_counts(dropna=False)
    pct = (counts / counts.sum() * 100).round(1)
    summary = pd.DataFrame({'n': counts, 'pct': pct})
    print(summary)
    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot(kind='bar', ax=ax, color=['#3b6fb6', '#d97757', '#888'][:len(counts)])
    for i, (n, p) in enumerate(zip(counts, pct)):
        ax.text(i, n, f'{n}\n({p}%)', ha='center', va='bottom')
    ax.set_ylabel('N sujetos')
    ax.set_title('Distribución por sexo — cohorte ATLAS R2.0')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / '4_2_sexo.png', dpi=120)
    plt.show()

## 5. EDA univariado · Sitio / cohorte (riesgo de sesgo multi-sitio)

**Pregunta:** ¿cuántos sitios / cohortes contribuyen y cuán balanceada está la representación? Un sitio dominante puede sesgar tanto el modelo como la generalización.

In [ ]:
if COL_COHORT is None:
    print('No se detectó columna de cohorte/sitio.')
else:
    counts = df[COL_COHORT].value_counts(dropna=False)
    print(f'Sitios/cohortes únicos: {counts.shape[0]}')
    print(counts.head(20))
    fig, ax = plt.subplots(figsize=(10, max(3, 0.25 * len(counts))))
    counts.sort_values().plot(kind='barh', ax=ax)
    ax.set_xlabel('N sujetos')
    ax.set_title('Distribución por sitio/cohorte — ATLAS R2.0')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'figures' / '5_cohorte.png', dpi=120)
    plt.show()

## 6. EDA univariado · Características de la lesión

El metadata oficial de ATLAS R2.0 **no incluye volumen de lesión precomputado**. Lo calculamos a partir de las máscaras NIfTI.

Para mantener tiempos razonables en la sesión del 6 de mayo, en este notebook trabajamos sobre una **muestra aleatoria de `SAMPLE_N` sujetos**. El cómputo full sobre los ~955 sujetos se hace en `02_features_lesion_full.ipynb` (Día 2).

In [ ]:
# Localizamos todas las máscaras de lesión (BIDS-like).
mask_files = list(DATA_ROOT.rglob('*lesion_mask.nii.gz')) + list(DATA_ROOT.rglob('*lesion_mask.nii'))
print(f'Máscaras encontradas: {len(mask_files)}')
if mask_files:
    print('Ejemplo:', mask_files[0])

In [ ]:
# Instalación defensiva de nibabel y scipy si no están disponibles.
try:
    import nibabel as nib
    from scipy import ndimage as ndi
except ModuleNotFoundError:
    print('Instalando nibabel y scipy ...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel', 'scipy'])
    import nibabel as nib
    from scipy import ndimage as ndi

def lesion_features(mask_path):
    """Devuelve dict con features cuantitativas de una máscara binaria de lesión."""
    img = nib.load(str(mask_path))
    data = np.asanyarray(img.dataobj) > 0
    voxel_mm3 = float(np.prod(img.header.get_zooms()[:3]))
    n_voxels  = int(data.sum())
    volume_mm3 = n_voxels * voxel_mm3
    # Lateralidad: fracción de voxels en la mitad izquierda del volumen.
    if n_voxels > 0:
        x_idx = np.where(data)[0]
        mid_x = data.shape[0] / 2
        frac_left = float((x_idx < mid_x).mean())
    else:
        frac_left = np.nan
    if   n_voxels == 0: lateralidad = 'sin_lesion'
    elif frac_left >= 0.85: lateralidad = 'izquierda'
    elif frac_left <= 0.15: lateralidad = 'derecha'
    else: lateralidad = 'bilateral'
    # Componentes conectados (medida de focalidad).
    n_components = int(ndi.label(data)[1]) if n_voxels > 0 else 0
    # Subject ID a partir del path (BIDS-like).
    m = re.search(r'sub-([A-Za-z0-9]+)', str(mask_path))
    subject_id = m.group(1) if m else mask_path.stem
    return {
        'subject_id': subject_id,
        'mask_path': str(mask_path),
        'voxel_mm3': voxel_mm3,
        'n_voxels_lesion': n_voxels,
        'volume_mm3': volume_mm3,
        'volume_ml': volume_mm3 / 1000.0,
        'frac_left': frac_left,
        'lateralidad': lateralidad,
        'n_components': n_components,
    }

print('lesion_features() listo.')

In [ ]:
# Cómputo sobre una muestra aleatoria.
rng = np.random.default_rng(RANDOM_SEED)
n = min(SAMPLE_N, len(mask_files))
sample_files = list(rng.choice(mask_files, size=n, replace=False))
print(f'Computando features sobre {n} máscaras ...')

rows = []
for i, p in enumerate(sample_files, 1):
    try:
        rows.append(lesion_features(p))
    except Exception as e:
        print(f'  [{i}/{n}] ERROR en {p.name}: {e}')
    if i % 10 == 0:
        print(f'  {i}/{n}')

feat = pd.DataFrame(rows)
feat.to_csv(OUTPUT_DIR / 'tables' / 'lesion_features_sample.csv', index=False)
print('Listo. Features computadas:')
feat.describe(include='all').T

### 6.1 Volumen de lesión (candidato a variable objetivo de regresión)

**Pregunta:** ¿qué forma tiene la distribución del volumen de lesión, y es viable como variable objetivo de regresión?

In [ ]:
vol = feat['volume_ml']
print(vol.describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(vol, bins=25, edgecolor='white')
axes[0].set_xlabel('Volumen de lesión (mL)')
axes[0].set_ylabel('N sujetos')
axes[0].set_title('Volumen — escala lineal')

vol_pos = vol[vol > 0]
axes[1].hist(np.log10(vol_pos), bins=25, edgecolor='white', color='#3b6fb6')
axes[1].set_xlabel(r'$\log_{10}$ Volumen (mL)')
axes[1].set_ylabel('N sujetos')
axes[1].set_title('Volumen — escala log10')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / '6_1_volumen_lesion.png', dpi=120)
plt.show()

print('\nObservación esperada: distribución típicamente log-normal —')
print('justifica una transformación log antes de modelar como regresión.')

### 6.2 Lateralidad (candidato a variable objetivo de clasificación)

**Pregunta:** ¿qué tan balanceadas están las clases izquierda / derecha / bilateral?

In [ ]:
lat = feat['lateralidad'].value_counts()
lat_pct = (lat / lat.sum() * 100).round(1)
print(pd.DataFrame({'n': lat, 'pct': lat_pct}))

fig, ax = plt.subplots(figsize=(6, 4))
lat.plot(kind='bar', ax=ax, color=['#3b6fb6', '#d97757', '#7a9c5d', '#888'])
for i, (n, p) in enumerate(zip(lat, lat_pct)):
    ax.text(i, n, f'{n}\n({p}%)', ha='center', va='bottom')
ax.set_ylabel('N sujetos')
ax.set_title('Lateralidad de la lesión — muestra')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / '6_2_lateralidad.png', dpi=120)
plt.show()

### 6.3 Número de componentes conectados (focalidad)

**Pregunta:** ¿las lesiones son típicamente focales (1 componente) o multifocales? Esto afecta la dificultad de la segmentación.

In [ ]:
nc = feat['n_components']
print(nc.describe())
fig, ax = plt.subplots(figsize=(7, 4))
nc.plot(kind='hist', bins=range(0, int(nc.max()) + 2), ax=ax, edgecolor='white')
ax.set_xlabel('N componentes conectados')
ax.set_ylabel('N sujetos')
ax.set_title('Focalidad de la lesión — muestra')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / '6_3_componentes.png', dpi=120)
plt.show()

## 7. Variable objetivo · resumen de las 3 candidatas

Tabla resumen para decidir el target en grupo.

In [ ]:
summary = pd.DataFrame([
    {
        'candidata': 'Volumen de lesión (mL)',
        'tipo_tarea': 'regresión',
        'n_obs': int(feat['volume_ml'].notna().sum()),
        'rango': f"{feat['volume_ml'].min():.2f} – {feat['volume_ml'].max():.2f}",
        'nota': 'Distribución log-normal probable; transformar antes de modelar.'
    },
    {
        'candidata': 'Máscara voxel-wise',
        'tipo_tarea': 'segmentación 3D',
        'n_obs': len(mask_files),
        'rango': 'binaria por voxel',
        'nota': 'Tarea canónica de ATLAS; requiere DL (U-Net 3D).'
    },
    {
        'candidata': 'Lateralidad',
        'tipo_tarea': 'clasificación 3-clases',
        'n_obs': int(feat['lateralidad'].notna().sum()),
        'rango': 'izquierda / derecha / bilateral',
        'nota': 'Baseline simple; revisar balance.'
    },
])
summary.to_csv(OUTPUT_DIR / 'tables' / 'target_options_summary.csv', index=False)
summary

## 8. Conclusiones del EDA univariado y siguiente paso

**Hallazgos a llenar tras correr el notebook (cada bullet es para la presentación final):**

- *Cohorte:* N total = ___, % faltantes en columnas clave = ___.
- *Edad:* rango binado ___, mediana del bin ___.
- *Sexo:* % M / % F = ___, balance ___.
- *Sitio:* ___ sitios; el sitio más representado aporta ___% (riesgo de sesgo multi-sitio: ___).
- *Volumen de lesión:* mediana = ___ mL, IQR = ___; distribución log-normal: sí/no.
- *Lateralidad:* izquierda ___% / derecha ___% / bilateral ___%.
- *Focalidad:* ___% de sujetos con 1 sólo componente.

**Decisión de variable objetivo (a tomar en grupo el 7 de mayo):** ___

**Riesgos del pipeline identificados en este EDA:**

1. **Edad binada en intervalos de 5 años** → restringe la regresión sobre edad.
2. **Posible desbalance de sitio** → si modelamos, estratificar el split por sitio.
3. **Volumen log-normal** → aplicar `log1p` antes de regresión.
4. **Data leakage potencial** → el split *debe* ser por `subject_id`, no por archivo (algunos sujetos tienen múltiples sesiones).
5. **Generalización a Guatemala** → ATLAS no incluye sitios latinoamericanos; honestidad sobre la transferencia.

**Compromiso para la sesión del 7 de mayo (mínimo según rúbrica):**

- Cómputo full de features de lesión sobre los ~955 sujetos.
- EDA bivariado: volumen vs edad, volumen vs sexo, lateralidad vs sitio.
- Primer baseline corriendo end-to-end (regresión logística / lineal sobre features tabulares).
- Métricas iniciales reportadas con CV estratificada por sitio.